# Agentic AI Fundamentals

## Notebook 03: Building Safe Tools and a Tool Registry

A model becomes operationally useful when it can request capabilities outside its own text generation. Those capabilities might retrieve documents, perform calculations, query services, or update external systems. Exposing them safely requires more than placing Python functions in a dictionary.

A simple way to think about this relationship:

> **LLM = the brain**  
>
> **Tools = the hands / capabilities the brain can use**  
>
> **Agent = the system that decides which tool to use and when**

```
        ┌───────────────┐
        │     USER      │
        └───────────────┘
                │
                │  "What's the weather in Ankara today?"
                ▼
        ┌───────────────┐
        │  LLM / AGENT  │
        └───────────────┘
                │
                │  Decides to call get_weather("Ankara")
                ▼
        ┌───────────────┐
        │ WEATHER TOOL  │
        └───────────────┘
                │
                │  Temperature: 18°C
                │  Rain probability: 80%
                ▼
        ┌───────────────┐
        │  LLM / AGENT  │
        └───────────────┘
                │
                │  "It's 18°C in Ankara with an
                │   80% chance of rain today."
                ▼
        ┌───────────────┐
        │     USER      │
        └───────────────┘
```

In this notebook, we will build a small tool layer that combines executable functions with descriptions, Pydantic argument schemas, risk metadata, and a controlled registry. The result will be independent of any particular model provider, which means we can reuse it when we connect a real API in the next notebook.

## Learning objectives

By the end of the notebook, you should be able to:

- identify the parts of a well-defined tool;
- validate tool arguments before function execution;
- register tools without allowing duplicate names;
- return structured observations for success and failure;
- require approval for side-effecting tools; and
- generate provider-ready JSON schemas from Pydantic models.

The examples use Pydantic 2 and the standard library. No model API key is required yet.

## 1. What belongs in a tool definition?

A useful tool definition has two audiences. The model needs a concise name, an accurate description, and an argument schema so it can decide whether and how to request the capability. The application needs the executable function, validation rules, and an execution policy.

We will represent these concerns together:

| Component | Purpose |
|---|---|
| Name | Stable identifier used in a model request |
| Description | Explains when the model should use the tool |
| Argument model | Validates the supplied parameters |
| Function | Implements the capability |
| Side-effect flag | Marks tools that change external state |

Good descriptions should distinguish nearby tools. “Search information” is vague; “Search the internal course notes for facts not present in the conversation” gives the model a clearer selection rule.

## 2. Define two low-risk capabilities

The calculator performs exact arithmetic, while the knowledge search retrieves matching local notes. Their implementations are intentionally simple; the focus is the interface surrounding them.

In [ ]:
import re


def multiply(first_number: float, second_number: float) -> float:
    """Multiply two numbers."""

    return first_number * second_number


COURSE_NOTES = {
    "tool": "A tool is a capability exposed and controlled by application code.",
    "registry": "A tool registry is an explicit allowlist of executable capabilities.",
    "observation": "A tool result becomes an observation in the agent state.",
    "workflow": "A workflow follows developer-defined transitions between steps.",
}


def search_course_notes(query: str, max_results: int = 2) -> list[dict[str, str]]:
    """Return course notes ranked by simple keyword overlap."""

    query_tokens = set(re.findall(r"[a-z0-9]+", query.casefold()))
    scored_notes = []

    for topic, text in COURSE_NOTES.items():
        note_tokens = set(re.findall(r"[a-z0-9]+", f"{topic} {text}".casefold()))
        score = len(query_tokens & note_tokens)

        if score > 0:
            scored_notes.append((score, topic, text))

    scored_notes.sort(reverse=True)

    return [
        {"topic": topic, "text": text}
        for _, topic, text in scored_notes[:max_results]
    ]


print(multiply(12, 9))
print(search_course_notes("What is a tool registry?"))

## 3. Describe each tool's input contract

A separate Pydantic model makes every tool's arguments explicit. The models reject unexpected fields, and the retrieval schema limits the number of requested results so a model cannot accidentally request an unbounded response.

In [ ]:
from pydantic import BaseModel, ConfigDict, Field


class MultiplyArguments(BaseModel):
    model_config = ConfigDict(extra="forbid")

    first_number: float = Field(description="The first multiplication operand")
    second_number: float = Field(description="The second multiplication operand")


class SearchCourseNotesArguments(BaseModel):
    model_config = ConfigDict(extra="forbid")

    query: str = Field(
        min_length=3,
        max_length=300,
        description="A focused search query for the course notes",
    )
    max_results: int = Field(
        default=2,
        ge=1,
        le=5,
        description="Maximum number of notes to return",
    )

## 4. Create a reusable tool object

The `Tool` class stores the public contract and the private implementation together. Its `execute` method validates raw arguments before calling the function. The function only receives data that has passed the schema.

We also record whether a tool has side effects. This does not provide authorization by itself, but it gives the executor enough information to enforce a policy.

In [ ]:
from dataclasses import dataclass
from typing import Any, Callable

from pydantic import ValidationError


@dataclass(frozen=True)
class Tool:
    name: str
    description: str
    arguments_model: type[BaseModel]
    function: Callable[..., Any]
    has_side_effects: bool = False

    def execute(self, raw_arguments: dict[str, Any]) -> Any:
        validated_arguments = self.arguments_model.model_validate(raw_arguments)
        return self.function(**validated_arguments.model_dump())

    def as_function_schema(self) -> dict[str, Any]:
        return {
            "type": "function",
            "name": self.name,
            "description": self.description,
            "parameters": self.arguments_model.model_json_schema(),
        }

In [ ]:
multiply_tool = Tool(
    name="multiply",
    description=(
        "Multiply two numbers when an exact multiplication result is required. "
        "Do not use this tool for addition, division, or factual questions."
    ),
    arguments_model=MultiplyArguments,
    function=multiply,
)

search_tool = Tool(
    name="search_course_notes",
    description=(
        "Search the local Agentic AI course notes for definitions or facts "
        "that are not already available in the conversation."
    ),
    arguments_model=SearchCourseNotesArguments,
    function=search_course_notes,
)


print(multiply_tool.execute({"first_number": 24, "second_number": 17}))
print(search_tool.execute({"query": "Explain an observation"}))

The descriptions include both intended and unintended use. This reduces ambiguity when several tools are available. A description cannot guarantee correct selection, but it gives the model better information and makes later evaluation more meaningful.

## 5. Manage tools through a registry

The registry owns the application's available tool set. It prevents silent replacement of an existing tool and provides one place to list, retrieve, and export tool definitions.

In [ ]:
class ToolRegistry:
    def __init__(self) -> None:
        self._tools: dict[str, Tool] = {}

    def register(self, tool: Tool) -> None:
        if tool.name in self._tools:
            raise ValueError(f"A tool named '{tool.name}' is already registered")

        self._tools[tool.name] = tool

    def get(self, tool_name: str) -> Tool | None:
        return self._tools.get(tool_name)

    def list_names(self) -> list[str]:
        return sorted(self._tools)

    def function_schemas(self) -> list[dict[str, Any]]:
        return [tool.as_function_schema() for tool in self._tools.values()]


registry = ToolRegistry()
registry.register(multiply_tool)
registry.register(search_tool)

print("Available tools:", registry.list_names())

In [ ]:
try:
    registry.register(multiply_tool)
except ValueError as error:
    print("Duplicate rejected:", error)

Duplicate registration is treated as an error because silently replacing an implementation could change application behavior without changing the public tool name.

## 6. Return structured observations

The model should receive a predictable result whether execution succeeds or fails. The executor below checks registration, approval requirements, argument validity, and runtime errors. It returns data rather than exposing raw exceptions to the rest of the agent loop.

In [ ]:
def execute_registered_tool(
    registry: ToolRegistry,
    tool_name: str,
    raw_arguments: dict[str, Any],
    approved: bool = False,
) -> dict[str, Any]:
    tool = registry.get(tool_name)

    if tool is None:
        return {
            "type": "tool_observation",
            "success": False,
            "tool_name": tool_name,
            "error_type": "unknown_tool",
            "error": f"Tool is not registered: {tool_name}",
        }

    if tool.has_side_effects and not approved:
        return {
            "type": "tool_observation",
            "success": False,
            "tool_name": tool_name,
            "error_type": "approval_required",
            "error": "This tool requires explicit approval before execution",
        }

    try:
        result = tool.execute(raw_arguments)
    except ValidationError as error:
        return {
            "type": "tool_observation",
            "success": False,
            "tool_name": tool_name,
            "error_type": "invalid_arguments",
            "error": error.errors(),
        }
    except Exception as error:
        return {
            "type": "tool_observation",
            "success": False,
            "tool_name": tool_name,
            "error_type": "execution_error",
            "error": f"{type(error).__name__}: {error}",
        }

    return {
        "type": "tool_observation",
        "success": True,
        "tool_name": tool_name,
        "result": result,
    }

In [ ]:
requests = [
    (
        "multiply",
        {"first_number": 8, "second_number": 7},
    ),
    (
        "multiply",
        {"first_number": 8, "second_numbr": 7},
    ),
    (
        "invented_tool",
        {},
    ),
]

for tool_name, arguments in requests:
    observation = execute_registered_tool(
        registry,
        tool_name,
        arguments,
    )
    print(observation)
    print("-" * 70)

Structured error types allow the control loop to respond differently. Invalid arguments might justify one constrained correction attempt. An unknown tool usually indicates that the model selected a capability outside the advertised set. An approval requirement should pause rather than be bypassed through automatic retry.

## 7. Side-effecting tools need a stronger boundary

To demonstrate the policy, we will create a simulated notification tool. It only returns a string, but we mark it as side-effecting because a real implementation would communicate with an external recipient.

In [ ]:
class NotificationArguments(BaseModel):
    model_config = ConfigDict(extra="forbid")

    recipient: str = Field(min_length=3, max_length=100)
    message: str = Field(min_length=1, max_length=500)


def send_notification(recipient: str, message: str) -> str:
    return f"Notification sent to {recipient}: {message}"


notification_tool = Tool(
    name="send_notification",
    description="Send a notification to an external recipient after user approval.",
    arguments_model=NotificationArguments,
    function=send_notification,
    has_side_effects=True,
)

registry.register(notification_tool)

notification_arguments = {
    "recipient": "course-admin",
    "message": "The tutorial validation completed successfully.",
}

without_approval = execute_registered_tool(
    registry,
    "send_notification",
    notification_arguments,
)

with_approval = execute_registered_tool(
    registry,
    "send_notification",
    notification_arguments,
    approved=True,
)

print("Without approval:", without_approval)
print("With approval   :", with_approval)

In a real application, the `approved` value must come from trusted application state or a human-confirmation interface. The model must never be able to approve its own request by adding an argument such as `approved=true`.

## 8. Export provider-ready tool schemas

Model APIs need tool names, descriptions, and JSON-compatible parameter schemas. Our registry can generate these definitions without exposing the Python functions themselves.

In [ ]:
import json


provider_tools = registry.function_schemas()
print(json.dumps(provider_tools, indent=2))

Different providers may wrap function definitions in slightly different outer structures, but the core information is consistent. In the next notebook, we will add a thin provider adapter instead of changing the tool implementation.

## 9. Exercise: add a word-count tool

Create and register a `count_words` tool that:

- accepts a `text` argument between 1 and 2,000 characters;
- rejects unexpected arguments;
- returns the number of whitespace-separated words;
- has no side effects; and
- includes a description that distinguishes it from document search.

Execute it through `execute_registered_tool` with the sentence `"Reliable agents expose a small and explicit action space."` The expected result is `9`.

In [ ]:
# Define CountWordsArguments.


# Define the count_words function.


# Create a Tool, register it, and execute it through the registry.

<details>
<summary><strong>Reveal one possible solution</strong></summary>

```python
class CountWordsArguments(BaseModel):
    model_config = ConfigDict(extra="forbid")

    text: str = Field(min_length=1, max_length=2_000)


def count_words(text: str) -> int:
    return len(text.split())


count_words_tool = Tool(
    name="count_words",
    description=(
        "Count whitespace-separated words in text supplied directly by the user. "
        "Do not use this tool to search documents or retrieve missing content."
    ),
    arguments_model=CountWordsArguments,
    function=count_words,
)

registry.register(count_words_tool)

exercise_observation = execute_registered_tool(
    registry,
    "count_words",
    {
        "text": "Reliable agents expose a small and explicit action space."
    },
)

print(exercise_observation)
```

</details>

## Notebook summary

We built a provider-neutral tool layer containing executable functions, Pydantic argument models, descriptions, side-effect metadata, and a registry. The executor rejects unknown tools and invalid arguments, converts failures into structured observations, and blocks side-effecting tools until approval is supplied by trusted application code.

The next notebook will connect this registry to a real language-model API. The model will receive the exported tool schemas, choose whether to request a function, and return structured arguments. Our application—not the model—will continue to own validation and execution.